<a href="https://colab.research.google.com/github/21f1000895-png/My_first_repo/blob/main/workshop_langchain_rag2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain langchain-core langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
!pip install -q langchain-groq langchain-huggingface sentence-transformers faiss-cpu pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.1/405.1 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.2 MB/s eta 0:00:00


In [6]:
from google.colab import userdata
from langchain_groq import ChatGroq

llm = ChatGroq (
    model = "openai/gpt-oss-120b",
    api_key = userdata.get("Groq_api_key")
)

In [8]:
response = llm.invoke("what is langchain? answer in one line only")
response.content

'LangChain is a framework for building applications that combine large language models with external data sources, tools, and custom logic.'

In [9]:
response = llm.invoke("what new product launched by acme inc in 2026?")
response.content

'I’m not aware of any specific product that Acme\u202fInc. announced or released in 2026. My training data only goes up through 2024, and I don’t have real‑time browsing capability to look up events that occurred after that point.\n\nIf you need the most up‑to‑date information, I recommend checking one of the following sources:\n\n| Source | How to use it |\n|--------|---------------|\n| **Acme\u202fInc.\u202fPress Releases** | Visit the company’s official website (usually a “News” or “Press” section) and filter by date or year. |\n| **Business News Websites** | Sites like Bloomberg, Reuters, or TechCrunch often cover major product launches. Use their search function with keywords like “Acme\u202fInc. 2026 product launch.” |\n| **SEC Filings (if publicly traded)** | Look at the company’s Form\u202f10‑K, 10‑Q, or 8‑K filings for any mention of new product introductions. |\n| **Social Media** | Check Acme’s official LinkedIn, Twitter, or Instagram accounts; companies often announce new p

#Rag
steps:
1. load the data for retrieval.
2. Split the data or documents.
3. produce the embeddings for each of the splits.
4. store the embeddings in a vecator db
5. create a retriever to retrive the embeddings from the vector db
6. combine the llm and the retriever and produce the results.

In [10]:
#document
from langchain_core.documents import Document
doc1=Document(page_content = "this is my document. it is completely random.", metadata={"source": "first_file"})
doc1


Document(metadata={'source': 'first_file'}, page_content='this is my document. it is completely random.')

Ways of loading data.


In [13]:
#load text file
from langchain_community.document_loaders import TextLoader
loader = TextLoader("RAG_file.txt")
doc1 = loader.load()
doc1

[Document(metadata={'source': 'RAG_file.txt'}, page_content='Acme Inc. was founded in 1987 in Helsinki, Finland. It specializes in anti-gravity footwear and rocket-powered pogo sticks.\n\nIn 2024, Acme released a new product line: "Jet Sneakers", designed for low-orbit recreational use.\n\nAs per internal policy, Acme\'s HR team meets every 2 weeks to assess wellness metrics of staff based on holographic surveys.\n')]

In [18]:
#load a pdf file
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("RAG_file.pdf")
docu2 = loader.load()
docu2


[Document(metadata={'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20250730135006', 'source': 'RAG_file.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Acme Inc. was founded in 1987 in Helsinki, Finland. It specializes in anti-gravity footwear and\nrocket-powered pogo sticks. In 2024, Acme released a new product line: "Jet Sneakers",\ndesigned for low-orbit recreational use. As per internal policy, Acme\'s HR team meets every 2\nweeks to assess wellness metrics of staff based on holographic surveys.')]

In [ ]:
#CSVLoader etc, you can add all docus together too by adding


#Split


In [28]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
#recursive character text splitter splits word by word repecting word boundary.
splitter = RecursiveCharacterTextSplitter(chunk_size = 50, chunk_overlap= 15)
chunks = splitter.split_documents(doc1)
chunks

[Document(metadata={'source': 'RAG_file.txt'}, page_content='Acme Inc. was founded in 1987 in Helsinki,'),
 Document(metadata={'source': 'RAG_file.txt'}, page_content='in Helsinki, Finland. It specializes in'),
 Document(metadata={'source': 'RAG_file.txt'}, page_content='specializes in anti-gravity footwear and'),
 Document(metadata={'source': 'RAG_file.txt'}, page_content='footwear and rocket-powered pogo sticks.'),
 Document(metadata={'source': 'RAG_file.txt'}, page_content='In 2024, Acme released a new product line: "Jet'),
 Document(metadata={'source': 'RAG_file.txt'}, page_content='line: "Jet Sneakers", designed for low-orbit'),
 Document(metadata={'source': 'RAG_file.txt'}, page_content='for low-orbit recreational use.'),
 Document(metadata={'source': 'RAG_file.txt'}, page_content="As per internal policy, Acme's HR team meets"),
 Document(metadata={'source': 'RAG_file.txt'}, page_content='HR team meets every 2 weeks to assess wellness'),
 Document(metadata={'source': 'RAG_file.tx

In [31]:
print(splitter._chunk_size)
print(splitter._chunk_overlap)

50
15


#Embeddings
converting the chunks into embeddins. sentence tranformers are used to create embeddings


In [33]:
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [39]:
vectors = embedding_model.embed_documents([doc.page_content for doc in chunks])
len(vectors)


11

In [38]:
len(vectors[1])

384

#store embeddings in vectorDB

In [40]:
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(chunks, embedding_model)


In [41]:
vectorstore

In [42]:
retrieved_docs = vectorstore.similarity_search("What was the launch date?", k = 2)
retrieved_docs

[Document(id='899a0a11-417e-40de-b21c-fbd6cbfe4176', metadata={'source': 'RAG_file.txt'}, page_content='In 2024, Acme released a new product line: "Jet'),
 Document(id='c419466a-1477-4849-a78d-b74658ba2e75', metadata={'source': 'RAG_file.txt'}, page_content='Acme Inc. was founded in 1987 in Helsinki,')]

# Create Retriever

In [53]:
retriever = vectorstore.as_retriever(search_kwargs = {"k": 2})

In [54]:
retriever.invoke("what was the launch date?")

[Document(id='899a0a11-417e-40de-b21c-fbd6cbfe4176', metadata={'source': 'RAG_file.txt'}, page_content='In 2024, Acme released a new product line: "Jet'),
 Document(id='c419466a-1477-4849-a78d-b74658ba2e75', metadata={'source': 'RAG_file.txt'}, page_content='Acme Inc. was founded in 1987 in Helsinki,')]

# Create RAG chain

In [55]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [56]:
def format_docs(docs):
  return "\n\n".join(doc.page_content for doc in docs)

In [57]:
template = """
Use the following context to answer the question.
Use three sentences maximum and keep the answer concise.

Context: {context}

Question: {question}

Answer:
"""

prompt = PromptTemplate.from_template(template)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [58]:
query = "What new product line was launched in 2024 in Acme Inc?"

response = rag_chain.invoke(query)
response

'Acme Inc. launched the “Jet” product line in 2024.'

In [61]:
query = "When and where was Acme Inc. founded?"

response = rag_chain.invoke(query)
response

'Acme Inc. was founded in 1987 in Helsinki.'